<a href="https://colab.research.google.com/github/BurroClurse/BrownBag/blob/main/BrownBagMatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brown Bag Matching Check: Wrong Joins and Wrong Splits

Synthetic and public data only (Synthea, RxNav, openFDA). Never upload real patient records.

## Setup: Libraries, Java and Node

Installs what Colab may lack. Java runs Synthea; Node runs Brown Bag's own matching code. Takes about a minute.

In [ ]:
N_PATIENTS = 300        # how many synthetic patients Synthea makes
YEARS = 3               # years of each patient's history to keep; more years make much bigger files
SEED = 42               # same seed, same patients, so runs can be compared
BRANCH = "main"         # the branch whose matching code is tested

!pip -q install pandas scikit-learn requests
!which java || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null)
!node -e "process.exit(Number(process.versions.node.split('.')[0]) >= 20 ? 0 : 1)" || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get -qq install -y nodejs > /dev/null)
!java -version 2>&1 | head -1
!node --version

import json, re, time, random, difflib, glob, os
import requests
import pandas as pd
random.seed(SEED)

## Step 1: Get Brown Bag's Matching Code

A copy of the repo. The notebook calls Brown Bag's own functions, so it tests the real rules.

In [ ]:
!rm -rf BrownBag && git clone -q --depth 1 -b {BRANCH} https://github.com/BurroClurse/BrownBag.git
!ls BrownBag/js/reconcile.js BrownBag/js/meds.js

## Step 2: Make Synthetic Patients with Synthea

Synthea writes one FHIR R4 Bundle per made-up patient, with real RxNorm medicines. No real people. YEARS (cell 1) keeps the files small enough for Colab to read quickly.

In [ ]:
!test -f synthea.jar || wget -q -O synthea.jar https://github.com/synthetichealth/synthea/releases/download/master-branch-latest/synthea-with-dependencies.jar
!rm -rf synthea_out && java -jar synthea.jar -p {N_PATIENTS} -s {SEED} --exporter.years_of_history={YEARS} --exporter.baseDirectory=./synthea_out > synthea.log 2>&1; tail -2 synthea.log
bundles = [f for f in glob.glob("synthea_out/fhir/*.json") if "Information" not in f]
len(bundles)

## Step 3: Read Every Medicine and Person

One row per patient and RxNorm product. Expect a few thousand rows for 300 patients.

In [ ]:
RXNORM = "http://www.nlm.nih.gov/research/umls/rxnorm"
people, meds = {}, []
for n, path in enumerate(bundles, 1):
    if n % 25 == 0:
        print("read", n, "of", len(bundles), flush=True)
    entries = [e["resource"] for e in json.load(open(path))["entry"]]
    patient = next(r for r in entries if r["resourceType"] == "Patient")
    people[patient["id"]] = patient
    seen = set()
    for r in entries:
        if r["resourceType"] != "MedicationRequest" or "medicationCodeableConcept" not in r:
            continue
        coding = next((c for c in r["medicationCodeableConcept"].get("coding", []) if c.get("system") == RXNORM), None)
        if not coding or coding["code"] in seen:
            continue
        seen.add(coding["code"])
        meds.append({"patient": patient["id"], "rxcui": coding["code"], "name": coding.get("display", ""), "resource": r})

med_df = pd.DataFrame([{k: v for k, v in m.items() if k != "resource"} for m in meds])
print(len(people), "patients,", len(med_df), "medicines,", med_df.rxcui.nunique(), "different products")
med_df.head()

## Step 4: Look Up Each Product in RxNorm

From RxNav (free, no key): each product's term type, dose form, ingredients and salts, plus up to 5 sibling products per ingredient. These are the answers the test is scored against. It prints its progress, and saves to rxnorm.json as it goes, so a re-run picks up where it stopped. About 5 to 10 minutes for 300 patients.

In [ ]:
RXNAV = "https://rxnav.nlm.nih.gov/REST"
cache = json.load(open("rxnorm.json")) if os.path.exists("rxnorm.json") else {}

from concurrent.futures import ThreadPoolExecutor
session = requests.Session()

def get(url):
    for attempt in range(3):
        try:
            time.sleep(0.1)           # 6 at a time, so well under RxNav's 20 requests a second
            return session.get(url, timeout=15).json()
        except Exception:
            time.sleep(2)
    return {}

def related(rxcui, ttys):
    groups = get(f"{RXNAV}/rxcui/{rxcui}/related.json?tty={'+'.join(ttys)}").get("relatedGroup", {}).get("conceptGroup", [])
    return {g["tty"]: g.get("conceptProperties", []) for g in groups}

def product(rxcui):
    if rxcui in cache:
        return cache[rxcui]
    props = get(f"{RXNAV}/rxcui/{rxcui}/properties.json").get("properties") or {}
    rel = related(rxcui, ["IN", "PIN", "DF"])
    cache[rxcui] = {
        "name": props.get("name", ""), "tty": props.get("tty", ""),
        "ingredients": sorted(c["rxcui"] for c in rel.get("IN", [])),
        "salts": sorted(c["rxcui"] for c in rel.get("PIN", [])),
        "form": (rel.get("DF") or [{}])[0].get("name", ""),
    }
    return cache[rxcui]

def look_up_all(codes, label):            # 6 lookups at a time, saving as it goes
    codes = [c for c in dict.fromkeys(codes) if c not in cache]
    with ThreadPoolExecutor(max_workers=6) as pool:
        for n, _ in enumerate(pool.map(product, codes), 1):
            if n % 50 == 0 or n == len(codes):
                print(label, n, "of", len(codes), flush=True)
                json.dump(cache, open("rxnorm.json", "w"))

look_up_all(sorted(med_df.rxcui.unique()), "products")
first_ingredients = sorted({cache[c]["ingredients"][0] for c in med_df.rxcui.unique() if cache.get(c, {}).get("ingredients")})
with ThreadPoolExecutor(max_workers=6) as pool:
    siblings = dict(zip(first_ingredients, pool.map(lambda i: [c["rxcui"] for c in related(i, ["SCD"]).get("SCD", [])][:5], first_ingredients)))
look_up_all([s for group in siblings.values() for s in group], "related products")
look_up_all([i for p in list(cache.values()) for i in p["ingredients"]], "ingredients")   # every ingredient gets its own entry too
json.dump(cache, open("rxnorm.json", "w"))
print(len(cache), "RxNorm concepts looked up")

## Step 5: Rough the Records Up the Way Real Sources Do

Each function copies a record and changes it the way a real source writes it (from Brown Bag's messy-data findings). The copy is still the same medicine or person, so the right answer is known.

In [ ]:
import copy

def concept(resource):
    return resource["medicationCodeableConcept"]

def ingredient_code_first(r):           # Epic: the ingredient's code listed before the product's
    r = copy.deepcopy(r); p = cache[concept(r)["coding"][0]["code"]]
    if p["ingredients"]:
        concept(r)["coding"].insert(0, {"system": RXNORM, "code": p["ingredients"][0], "display": cache[p["ingredients"][0]]["name"]})
    return r

ABBREVIATIONS = [(" Oral Tablet", " TAB"), (" Oral Capsule", " CAP"), ("Hydrochloride", "HCL"),
                 ("Extended Release", "ER"), (" MG", "MG")]
def caps_and_abbreviations(r):          # pharmacy systems: capitals, TAB, HCL, ER
    r = copy.deepcopy(r); text = concept(r)["coding"][0].get("display", "")
    for long, short in ABBREVIATIONS:
        text = text.replace(long, short)
    concept(r)["text"] = text.upper()
    return r

def text_only(r):                       # no code at all, only words
    r = caps_and_abbreviations(r); concept(r).pop("coding")
    return r

def leading_dot(r):                     # "0.5 MG" written ".5 mg"
    r = text_only(r); concept(r)["text"] = re.sub(r"\b0\.(\d)", r".\1", concept(r)["text"])
    return r

MED_ROUGHERS = [ingredient_code_first, caps_and_abbreviations, text_only, leading_dot]

def last_first_text(p):                 # Cerner: "LAST, FIRST" in name.text only
    p = copy.deepcopy(p); n = p["name"][0]
    p["name"] = [{"text": f"{n['family'].upper()}, {n['given'][0].upper()}"}]
    return p

def no_digits(p):                       # Synthea adds numbers to names; real sources don't
    p = copy.deepcopy(p)
    for n in p["name"]:
        n["family"] = re.sub(r"\d+", "", n.get("family", "")); n["given"] = [re.sub(r"\d+", "", g) for g in n.get("given", [])]
    return p

def datetime_birth_date(p):             # a time stuck on the birth date
    p = copy.deepcopy(p); p["birthDate"] = p["birthDate"] + "T00:00:00Z"
    return p

def nickname_first(p):                  # a nickname listed before the legal name
    p = copy.deepcopy(p); p["name"].insert(0, {"use": "nickname", "given": ["Sunny"]})
    return p

def birth_date_typo(p):                 # day and month swapped: same person, but Brown Bag should keep them apart
    p = copy.deepcopy(p); y, m, d = p["birthDate"].split("-")
    if int(d) <= 12 and d != m:
        p["birthDate"] = f"{y}-{d}-{m}"
    return p

PERSON_ROUGHERS = [last_first_text, no_digits, datetime_birth_date, nickname_first, birth_date_typo]
print(len(MED_ROUGHERS), "medicine and", len(PERSON_ROUGHERS), "person roughers")

## Step 6: Build Test Pairs with Known Answers

Each pair is two records and the right answer. "join" means Brown Bag should treat them as one medicine or person, "apart" means it must not.

In [ ]:
def request_for(rxcui):                 # a plain record of another product, for the pairs that must stay apart
    return {"resourceType": "MedicationRequest", "status": "active", "authoredOn": "2026-01-01",
            "medicationCodeableConcept": {"coding": [{"system": RXNORM, "code": rxcui, "display": cache[rxcui]["name"]}],
                                          "text": cache[rxcui]["name"]}}

def truth(a, b):                        # the right answer, from RxNorm: same ingredients, salts and dose form
    pa, pb = cache[a], cache[b]
    same = pa["ingredients"] == pb["ingredients"] and pa["salts"] == pb["salts"] and pa["form"] == pb["form"]
    return "join" if same else "apart"

pairs = []
for m in meds:
    p = cache.get(m["rxcui"])
    if not p or not p["ingredients"]:
        continue
    for rough in MED_ROUGHERS:          # the same record, roughed up: should join
        pairs.append({"kind": "medicine", "tag": rough.__name__, "truth": "join", "a": m["resource"], "b": rough(m["resource"])})
    for other in siblings.get(p["ingredients"][0], []):
        if other == m["rxcui"] or other not in cache:
            continue
        answer = truth(m["rxcui"], other) # another strength of the same thing joins; another salt or form stays apart
        for rough in [None, ingredient_code_first, text_only]:
            b = request_for(other) if rough is None else rough(request_for(other))
            pairs.append({"kind": "medicine", "tag": "sibling, " + (rough.__name__ if rough else "as written"), "truth": answer, "a": m["resource"], "b": b})

ids = list(people)
for pid in ids:
    for rough in PERSON_ROUGHERS:
        answer = "apart" if rough is birth_date_typo and rough(people[pid])["birthDate"] != people[pid]["birthDate"] else "join"
        pairs.append({"kind": "person", "tag": rough.__name__, "truth": answer, "a": people[pid], "b": rough(people[pid])})
    other = people[random.choice(ids)]
    if other["id"] != pid:              # a different person: must stay apart
        pairs.append({"kind": "person", "tag": "different person", "truth": "apart", "a": people[pid], "b": other})

for i, pair in enumerate(pairs):
    pair["id"] = i
with open("pairs.jsonl", "w") as f:
    f.writelines(json.dumps(p) + "\n" for p in pairs)
pd.DataFrame(pairs).groupby(["kind", "truth"]).size()

## Step 7: Ask Brown Bag's Own Code

A small Node script runs Brown Bag's real matching functions on every pair, with Brown Bag's own RxNorm ingredient and salt lookup, so it answers exactly as the app would. About a minute.

In [ ]:
open("ask_brown_bag.mjs", "w").write("""
import { readFileSync, writeFileSync } from "node:fs";
import { fromMedicationRequest } from "./BrownBag/js/meds.js";
import { matchOf, GROUPS, personOf, samePerson } from "./BrownBag/js/reconcile.js";
import { ingredientsFor } from "./BrownBag/js/rxnorm.js";
const pairs = readFileSync("pairs.jsonl", "utf8").trim().split("\\n").map(function (line) { return JSON.parse(line); });
const codes = new Set();
pairs.forEach(function (p) {
  if (p.kind !== "medicine") return;
  [p.a, p.b].forEach(function (r) { (r.medicationCodeableConcept.coding || []).forEach(function (c) { codes.add(c.code); }); });
});
const map = await ingredientsFor(Array.from(codes));   // the app's own lookup, salts included
const out = pairs.map(function (p) {
  if (p.kind === "person") return { id: p.id, answer: samePerson(personOf(p.a), personOf(p.b)) ? "join" : "apart", level: null };
  const level = matchOf(fromMedicationRequest(p.a, "A"), fromMedicationRequest(p.b, "B"), map);
  return { id: p.id, answer: GROUPS.indexOf(level) !== -1 ? "join" : "apart", level: level };
});
writeFileSync("brown_bag.jsonl", out.map(function (o) { return JSON.stringify(o); }).join("\\n"));
console.log("Brown Bag answered", out.length, "pairs, using", map.size, "ingredient lookups");
""")
!node ask_brown_bag.mjs

## Step 8: Score Brown Bag's Rules

Wrong joins (two different medicines or people made one) must be zero: that's the dangerous mistake. Wrong splits are the price of being careful.

In [ ]:
answers = pd.DataFrame([json.loads(l) for l in open("brown_bag.jsonl")])
results = pd.DataFrame(pairs)[["id", "kind", "tag", "truth"]].merge(answers, on="id")
results["outcome"] = results.apply(lambda r: "right" if r.truth == r.answer
                                   else ("WRONG JOIN" if r.answer == "join" else "wrong split"), axis=1)
print(results.outcome.value_counts().to_string())
results.pivot_table(index=["kind", "tag"], columns="outcome", values="id", aggfunc="count", fill_value=0)

## Step 9: Turn Each Pair into Numbers for a Model

Features a pharmacist would look at: same code, same ingredients, same strength, same form, release and salt words, and how alike the names are.

In [ ]:
def words(resource):
    c = resource.get("medicationCodeableConcept", {})
    return (c.get("text") or (c.get("coding") or [{}])[0].get("display", "")).lower()

def codes(resource):
    return [x["code"] for x in resource.get("medicationCodeableConcept", {}).get("coding", []) if x.get("system") == RXNORM]

STRENGTH = re.compile(r"(\d*\.?\d+)\s*(mg|mcg|ug|g|ml|unt|units?|iu|%)", re.I)
RELEASE = re.compile(r"\b(\d+ hr|er|xr|xl|sr|cr|dr|la|odt|extended|delayed)\b", re.I)
SALTS = re.compile(r"\b(hydrochloride|hcl|tartrate|succinate|sodium|potassium|maleate|besylate|sulfate|citrate)\b", re.I)
FORMS = re.compile(r"\b(tablet|tab|capsule|cap|solution|suspension|injection|patch|cream|ointment|inhaler|spray|drops?)\b", re.I)

def strengths(text):
    return sorted(round(float(v) * (0.001 if u.lower() in ("mcg", "ug") else 1), 6) for v, u in STRENGTH.findall(text))

def features(pair):
    a, b = pair["a"], pair["b"]
    ta, tb = words(a), words(b)
    ca, cb = codes(a), codes(b)
    ia = {i for c in ca for i in cache.get(c, {}).get("ingredients", [])}
    ib = {i for c in cb for i in cache.get(c, {}).get("ingredients", [])}
    first = lambda t: (re.findall(r"[a-z]+", t) or [""])[0]
    return {
        "both_coded": int(bool(ca and cb)),
        "a_code_in_b": int(bool(set(ca) & set(cb))),
        "same_ingredients": int(bool(ia) and ia == ib),
        "same_strengths": int(strengths(ta) == strengths(tb)),
        "same_release": int(sorted(RELEASE.findall(ta)) == sorted(RELEASE.findall(tb))),
        "same_salts": int({s.lower().replace("hcl", "hydrochloride") for s in SALTS.findall(ta)} ==
                          {s.lower().replace("hcl", "hydrochloride") for s in SALTS.findall(tb)}),
        "same_form_word": int({f.lower()[:3] for f in FORMS.findall(ta)} == {f.lower()[:3] for f in FORMS.findall(tb)}),
        "same_first_word": int(first(ta) == first(tb)),
        "name_alike": difflib.SequenceMatcher(None, ta, tb).ratio(),
    }

med_pairs = [p for p in pairs if p["kind"] == "medicine"]
X = pd.DataFrame([features(p) for p in med_pairs])
y = (pd.Series([p["truth"] for p in med_pairs]) == "join").astype(int)
X.describe().T[["mean", "min", "max"]]

## Step 10: Train a Model on the Known Answers

Gradient boosting learns which features mean "same medicine". Patients are split so the test set has people the model never saw.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report

groups = [p["a"].get("subject", {}).get("reference", "") for p in med_pairs]
train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X, y, groups))
model = GradientBoostingClassifier(random_state=SEED).fit(X.iloc[train], y.iloc[train])
predicted = model.predict(X.iloc[test])

print(confusion_matrix(y.iloc[test], predicted))
print(classification_report(y.iloc[test], predicted, target_names=["apart", "join"]))
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

## Step 11: Find Where the Model and the Rules Disagree

This is the testing aid. A pair the model is sure about, that Brown Bag's rules answer the other way, is a likely new kind of mess. A person reviews each one; the model never decides anything in the app.

In [ ]:
med_results = results[results.kind == "medicine"].reset_index(drop=True)
med_results["model_join"] = model.predict_proba(X)[:, 1]
sure = (med_results.model_join > 0.95) | (med_results.model_join < 0.05)
model_says = med_results.model_join.round().map({1: "join", 0: "apart"})
flagged = med_results[sure & (model_says != med_results.answer)].copy()
flagged["a_name"] = [words(med_pairs[i]["a"]) for i in flagged.index]
flagged["b_name"] = [words(med_pairs[i]["b"]) for i in flagged.index]
print(len(flagged), "pairs to review, in these groups:")
print(flagged.groupby(["tag", "truth", "answer"]).size().sort_values(ascending=False).to_string())
# Three examples from each group: one group is usually one gap in the rules
flagged.groupby(["tag", "truth", "answer"]).head(3)[["tag", "truth", "answer", "level", "model_join", "a_name", "b_name"]]

## Step 12: Try It on Real Product Names from openFDA

openFDA's NDC directory lists real labels from many manufacturers and repackagers. No answers here, so only the model's flags are shown, for a person to check.

In [ ]:
INGREDIENTS = ["metformin", "metoprolol", "lisinopril", "atorvastatin", "bupropion", "diltiazem", "levothyroxine", "methotrexate"]
products = []
for name in INGREDIENTS:
    data = get(f'https://api.fda.gov/drug/ndc.json?search=generic_name:"{name}"&limit=100').get("results", [])
    for d in data:
        strength = " / ".join(f"{i['name']} {i['strength']}" for i in d.get("active_ingredients", []))
        products.append({"ingredient": name, "text": f"{d.get('generic_name', '')} {strength} {d.get('dosage_form', '')}",
                         "rxcuis": d.get("openfda", {}).get("rxcui", [])})

def as_record(text):
    return {"resourceType": "MedicationRequest", "status": "active", "authoredOn": "2026-01-01",
            "medicationCodeableConcept": {"text": text}}

real_pairs = []
for name in INGREDIENTS:
    group = [p for p in products if p["ingredient"] == name][:30]
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            if group[i]["text"] != group[j]["text"]:
                real_pairs.append({"a": as_record(group[i]["text"]), "b": as_record(group[j]["text"]),
                                   "fda_says": "join" if set(group[i]["rxcuis"]) & set(group[j]["rxcuis"]) else "apart"})
RX = pd.DataFrame([features(p) for p in real_pairs])
real = pd.DataFrame({"a": [p["a"]["medicationCodeableConcept"]["text"] for p in real_pairs],
                     "b": [p["b"]["medicationCodeableConcept"]["text"] for p in real_pairs],
                     "fda_says": [p["fda_says"] for p in real_pairs],
                     "model_join": model.predict_proba(RX)[:, 1]})
print(len(products), "real products,", len(real_pairs), "pairs")
real[(real.model_join > 0.95) & (real.fda_says == "apart")].head(15)

## Step 13: Save the Flagged Cases for Brown Bag's Tests

Writes the flagged pairs in the shape Brown Bag's test fixtures use, then downloads the file. Put it in tests/fixtures/messy/generated/ and write a test for each case that turns out real.

In [ ]:
cases = [{"case": f"generated {int(r.id)}", "tag": r.tag, "truth": r.truth, "brown_bag": r.answer,
          "level": r.level, "model_join": round(float(r.model_join), 3),
          "a": med_pairs[i]["a"]["medicationCodeableConcept"], "b": med_pairs[i]["b"]["medicationCodeableConcept"]}
         for i, r in flagged.iterrows()]
json.dump({"made": time.strftime("%Y-%m-%d"), "patients": N_PATIENTS, "seed": SEED,
           "wrong_joins": int((results.outcome == "WRONG JOIN").sum()),
           "wrong_splits": int((results.outcome == "wrong split").sum()),
           "cases": cases}, open("brown-bag-generated-cases.json", "w"), indent=1)
print(len(cases), "cases saved")
try:
    from google.colab import files
    files.download("brown-bag-generated-cases.json")
except ImportError:
    print("Not in Colab: the file is brown-bag-generated-cases.json")